In [0]:

# Look through the files in the directory
dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/green/")


In [0]:
# Load the dataset
df = spark.read.csv("dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-12.csv.gz", header=True, inferSchema=True)

Inspect the data

In [0]:
# Show the schema
df.printSchema()

In [0]:
# Show some rows
df.show(5)

In [0]:
# Row & column counts
df.count()
len(df.columns)

In [0]:
# List column names
df.columns

Selecting columns

In [0]:
# Select specific columns
df.select("VendorID", "trip_distance", "fare_amount").show(5)

In [0]:
# Rename columns
df.select(
    df.VendorID.alias("vendor_id"),
    df.trip_distance.alias("distance")
).show(5)

Filtering rows

In [0]:
df.filter(df.trip_distance > 5).show(5)

In [0]:
# Multiple conditions
df.filter(
    (df.trip_distance > 5) & (df.fare_amount > 20)
).show(5)

In [0]:
# SQL-style filter
df.filter("trip_distance > 5 AND fare_amount > 20").show(5)

Adding + modifying columns

In [0]:
# Add a new calculated column
from pyspark.sql.functions import col, try_divide

df2 = df.withColumn(
    "fare_per_mile",
    try_divide(col("fare_amount"), col("trip_distance"))
)

df2.select("fare_amount", "trip_distance", "fare_per_mile").show(5)

In [0]:
# Conditional columns (CASE WHEN)
from pyspark.sql.functions import when

df3 = df.withColumn(
    "long_trip_flag",
    when(col("trip_distance") >= 10, "long").otherwise("short")
)

df3.select("trip_distance", "long_trip_flag").show(5)

Aggregations (GROUP BY)


In [0]:
# Average fare by vendor
df.groupBy("VendorID").avg("fare_amount").show()

In [0]:
# Multiple aggregations
from pyspark.sql.functions import avg, sum, count

df.groupBy("VendorID").agg(
    avg("fare_amount").alias("avg_fare"),
    sum("fare_amount").alias("total_fare"),
    count("*").alias("trip_count")
).show()

Sorting & limiting

In [0]:
# Order by
df.orderBy(df.fare_amount.desc()).show(5)

In [0]:
# Top 10 rows
df.orderBy("trip_distance", ascending=False).limit(10).show()

Handling nulls

In [0]:
# Count nulls per column
from pyspark.sql.functions import isnan, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
# Drop nulls
df.dropna().count()

In [0]:
# Fill nulls
df.fillna({"payment_type": 0})

Dates & timestamps

In [0]:
# Extract year / month / day
from pyspark.sql.functions import year, month, dayofmonth

df.withColumn("year", year("tpep_pickup_datetime")) \
  .withColumn("month", month("tpep_pickup_datetime")) \
  .withColumn("day", dayofmonth("tpep_pickup_datetime")) \
  .select("tpep_pickup_datetime", "year", "month", "day") \
  .show(5)

Joins

In [0]:
# Self Joins
df_alias1 = df.select("VendorID", "fare_amount")
df_alias2 = df.select("VendorID", "trip_distance")

joined = df_alias1.join(
    df_alias2,
    on="VendorID",
    how="inner"
)

joined.show(5)

Window functions

In [0]:
# Rank trips by fare per vendor
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.partitionBy("VendorID").orderBy(col("fare_amount").desc())

df.withColumn("rank", rank().over(window_spec)) \
  .select("VendorID", "fare_amount", "rank") \
  .show(10)

Spark SQL

In [0]:
# Register temp view
df.createOrReplaceTempView("taxi")

In [0]:
df_sql = spark.sql("""
SELECT VendorID,
       AVG(fare_amount) AS avg_fare
FROM taxi
GROUP BY VendorID
""")

df_sql.show()